# Set up imports

In [1]:
import logging
import numpy as np
from blastwave import BOOMClient, BOOMCredentialsError

In [2]:
logger = logging.getLogger(__name__)

# Create Boom Client

In [3]:
boom = BOOMClient()

# Try pinging server

In [4]:
try:
    boom.ping()
except BOOMCredentialsError as exc:
    logger.error("Error with querying BOOM using credentials")
    raise exc

# See list of available catalogs

In [5]:
boom.get_catalogs()

['2MASS_PSC',
 'CatWISE2020',
 'DESI_DR1',
 'GALEX',
 'Gaia_DR3',
 'LSDR10',
 'LSPSC',
 'LSST_alerts',
 'LSST_alerts_aux',
 'LSST_alerts_cutouts_20260519',
 'LS_DR10_PHOTOZ',
 'MPC_orbits',
 'NED',
 'PS1_DR2',
 'TNS',
 'VSX',
 'WINTER_alerts',
 'WINTER_alerts_aux',
 'WINTER_alerts_cutouts',
 'ZTF_alerts',
 'ZTF_alerts_aux',
 'ZTF_alerts_cutouts',
 'ZTF_alerts_cutouts_20260519',
 'ZTF_sso_baselines',
 'babamul_oauth_states',
 'babamul_pending_identities',
 'milliquas_v8']

# Check a catalog

In [6]:
catalog = "DESI_DR1"
# catalog="ZTF_alerts_aux"

In [7]:
assert catalog in boom.get_catalogs(), f"Catalog {catalog} not listed in catalogs"

In [8]:
entry_count = boom.get_entry_count(catalog)
print(f"Catalog {catalog} has {entry_count} entries")

Catalog DESI_DR1 has 23229349 entries


In [9]:
indexes = boom.get_catalog_indexes(catalog)
print(f"Catalog {catalog} has the following index columns: {indexes}")

Catalog DESI_DR1 has the following index columns: [{'key': {'_id': 1}, 'name': '_id_', 'v': 2}, {'key': {'coordinates.radec_geojson': '2dsphere'}, 'name': 'coordinates.radec_geojson_2dsphere', 'v': 2, '2dsphereIndexVersion': 3}]


In [10]:
example = boom.get_sample_data(catalog)
print(f"Example entry from {catalog}:\n")
example

Example entry from DESI_DR1:



{'_id': 39627939007955361,
 'ra': 207.92821387445312,
 'dec': 6.321187701342101,
 'survey': 'main',
 'program': 'dark',
 'z': 0.8986591117094143,
 'zerr': 9.932572456539864e-05,
 'zwarn': 0,
 'chi2': 7740.26944501698,
 'deltachi2': 152.01993865519762,
 'spectype': 'GALAXY',
 'zcat_nspec': 1,
 'coordinates': {'radec_geojson': {'type': 'Point',
   'coordinates': [27.92821387445312, 6.321187701342101]}}}

# Query a catalog

BOOM uses MongoDB type queries. You can query on any of the fields in the sample data, but it'll be faster if you query on indexes 

In [11]:
# Here's an example, let's take the 3 sources closest to the above object. Look up MongoDB queries for syntax.
ra, dec = example["ra"], example["dec"]

projection = { # Choose to only return some fields
    "source_name": 1,
    "ra": 1,
    "dec": 1,
    "spectype": 1,
    "z": 1,
}

In [12]:
boom.cone_search(
    ra=ra, dec=dec,
    radius_arcsec=50.,
    catalog=catalog,
    limit=10,
    projection=projection
)

[{'_id': 39627939007955361,
  'ra': 207.92821387445312,
  'dec': 6.321187701342101,
  'z': 0.8986591117094143,
  'spectype': 'GALAXY'},
 {'_id': 39627939007955510,
  'ra': 207.93536045684883,
  'dec': 6.330464990831612,
  'z': 0.20720505812348392,
  'spectype': 'GALAXY'}]

We can also set custom filters for our query, in MongoDB language.

In [13]:
filter_query = {
    "z": {"$lt": 0.5}
}

boom.cone_search(
    ra=ra, dec=dec,
    radius_arcsec=200.,
    catalog=catalog,
    limit=10.,
    projection=projection,
    filter_query=filter_query
)

[{'_id': 39627939007955510,
  'ra': 207.93536045684883,
  'dec': 6.330464990831612,
  'z': 0.20720505812348392,
  'spectype': 'GALAXY'},
 {'_id': 39627939003765891,
  'ra': 207.90440953445815,
  'dec': 6.323050521714894,
  'z': -0.00034154090530624434,
  'spectype': 'STAR'},
 {'_id': 39627939007955247,
  'ra': 207.921273306233,
  'dec': 6.345891891219426,
  'z': 3.749985632279858e-05,
  'spectype': 'STAR'},
 {'_id': 39627939007955539,
  'ra': 207.9366235935039,
  'dec': 6.294985324497663,
  'z': -0.00014096319317550635,
  'spectype': 'STAR'},
 {'_id': 39627939007955681,
  'ra': 207.94428771924788,
  'dec': 6.293797944632686,
  'z': 0.24093433143587892,
  'spectype': 'GALAXY'},
 {'_id': 39627939003765749,
  'ra': 207.89703530773102,
  'dec': 6.3282937711226745,
  'z': 7.308346838969636e-05,
  'spectype': 'STAR'},
 {'_id': 39627939003765782,
  'ra': 207.8989026019089,
  'dec': 6.30641593989187,
  'z': 0.061080512016625205,
  'spectype': 'GALAXY'},
 {'_id': 39627939003765707,
  'ra': 207.

# See the full BOOM API documentation at https://api.kaboom.caltech.edu/docs 
# See MongoDB documentation at https://www.mongodb.com/docs/manual/reference/mql/query-predicates